# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 以init_chat_model为例
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})


rprint(response)

for message in response["messages"]:
    message.pretty_print()

## 2、系统提示词的设置

使用 system_prompt 参数进行设置，可以是 str, 也可以是 SystemMessage

输出的 messages 里面并不会有 SystemMessage。

举例1：并行调用

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 1.导入模型
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)
## TavilySearch 不传 tavily_api_key 也能运行，是因为它内置了自动从环境变量读取密钥的机制。代码中的 load_dotenv(override=True) 将 .env 文件里的 TAVILY_API_KEY 加载到了系统环境中，TavilySearch 在初始化时会自动获取这个值，所以不需要手动传参。

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

**为什么会出现连续两条 ToolMessage?**

- 模型在一次回复里发起了多个 tool_calls（并行工具调用）。

    - 一条 AIMessage 里可以包含 多个 tool_calls（OpenAI / DeepSeek 等都支持 parallel tool calls）。

    - 这里 AIMessage.tool_calls 里就一次性带了两个 tool_call。

        ```python
        tool_calls = [
            {  # 第一次
                'name': 'tavily_search',
                'args': {'query': '2024年诺贝尔物理学奖得主', ...},
                'id': 'call_00_gV6x3g3XCWDxEviUq7OS4935',
            },
            {  # 第二次
                'name': 'tavily_search',
                'args': {'query': '2024 Nobel Prize in Physics winners announcement', ...},
                'id': 'call_01_8KBqInWgbIjuemaWjNex9015',
            },
        ]
        ```
    - 说明大模型决定进行两次工具调用，一次用中文搜索，一次用英文搜索。

- 有几个 tool_call，Agent 就会执行几次工具，并紧接着追加几条 ToolMessage。

- 这些 ToolMessage 是“连续”的，因为它们都对应同一条 AIMessage 里的不同 tool_call_id。


**并行调用**

只进行一轮工具调用为例，可能并行调用工具，可能是同一个工具并行调用，也可能是不同的工具并行调用，判断是不是“并行调用”，**不看工具是不是同一个，而看这些 tool_call 是不是出现在同一条 `AIMessage` 里**。


**“并行”指的是模型侧，不代表运行时一定真并发**

- 模型侧：一次生成多个 tool_calls，这是并行工具调用。

- 执行侧：LangChain / LangGraph 的 ToolNode 可能用线程池或 asyncio.gather 并发执行，也可能顺序执行，取决于框架实现和工具类型（同步/异步）。

但无论执行是并发还是顺序，语义上都属于“同一批工具调用”，所以仍叫并行工具调用。

模型侧大模型可能会选择并行调用，也可能选择循环调用，这取决于具体情况。

举例2：循环调用

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

举例3：重试机制（也属循环调用）

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user", "content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

你遇到的 BadRequestError，核心原因是 DeepSeek API 目前不支持 ProviderStrategy 所使用的 json_schema 结构化输出格式。

**问题根源：ProviderStrategy 与 DeepSeek 的不兼容**

- 在你的代码中，ProviderStrategy(ContractInfo) 会指示 LangChain 尝试调用模型提供商的原生结构化输出功能（即 response_format: {"type": "json_schema", ...}）。

- 然而，DeepSeek 的 API 明确不支持 json_schema 这种响应格式，当收到此类请求时会直接拒绝并返回 400 错误。错误信息 This response_format type is unavailable now 正源于此。

详细解释：当传入 ProviderStrategy(ContractInfo) 时，**LangChain 会先将 ContractInfo 这个 Pydantic 模型内部转换为一个 JSON Schema 对象**，然后以 response_format: {"type": "json_schema", ...} 的形式发送给 DeepSeek API。DeepSeek 拒绝的是这个 API 层面的 json_schema 请求类型。

### 3.2 ToolStrategy策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)
print(type(response["structured_response"]))
print(response["structured_response"])

### 3.3 AutoStrategy / Type策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)
print(type(response["structured_response"]))
print(response["structured_response"])